# FEEDS — manuscript comparison table

Self-contained notebook (no `.py` import — everything is in these cells).

Produces the manuscript table:
- **AutoPET** — per-diagnosis rows (Dice/FN Vol on diseased diagnoses; FP Vol also on Negative)
- **Deep-PSMA** and **DH** — pooled rows

Two families of tests:
1. **Superiority** — FEEDS vs random (paired Wilcoxon + t). **FDR-corrected per metric, pooled across all datasets** (Benjamini–Hochberg via `statsmodels`).
2. **Non-inferiority** — FEEDS vs 100% labeled. Separate family, **not** included in the correction (✓/✗ flag).

Random iterations are summarized per case as both **mean** and **median** (switch with `SUMMARY`).

**Correction needs the whole family of p-values at once** (Holm/BH sort and threshold the full set), so the table is assembled first, then corrected. The number of tests per metric is just the row count, printed when you build the table.

## 0 · Imports & config

In [25]:
!pip install statsmodels

In [26]:
import os, json
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests   # correction package

# --- point at the results root (now that AutoPET is reachable too) ---
os.environ['nnUNet_results'] = '//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results'
assert 'nnUNet_results' in os.environ, "set os.environ['nnUNet_results'] first"

TRAINER  = "autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres"
METADATA = 'fdg_metadata.csv'

# ---- summary of the 5 random iterations: 'mean' or 'median' ----
SUMMARY    = 'mean'
# ---- multiplicity correction method: 'fdr_bh' (Benjamini-Hochberg) or 'holm' ----
CORRECTION = 'fdr_bh'
ALPHA      = 0.05

## 1 · CONFIG — datasets, SPLITs, arms

`CONFIG` keeps the familiar shape. **SPLIT is chosen per dataset here** — this is the only thing that changes between AutoPET / Deep-PSMA / DH.

- `layout='diagnosis'` → one row per disease (AutoPET block).
- `layout='pooled'` → a single pooled row (Deep-PSMA, DH).
- Arms: `feeds` (30%), `random` (5 iterations, pooled per case), `full` (100%).

In [27]:
CONFIG = {
    'AutoPET':  dict(split='test_predictions',        layout='diagnosis'),
    'DeepPSMA': dict(split='test_deep_psma_predictions',  layout='pooled'),
    'DH':       dict(split='DHMC_data_predictions',   layout='pooled'),
}

# arm -> (dataset number, folds, aggregation). Same mapping as before.
ARMS = dict(
    feeds =dict(dataset=111, folds=['fold_4'], agg='single'),
    random=dict(dataset=330, folds=['fold_0','fold_1','fold_2','fold_3','fold_4'], agg='iterations'),
    full  =dict(dataset=999, folds=['fold_9'], agg='single'),
)

# pre-specified non-inferiority margins (fix a priori with clinical co-authors)
MARGINS = {'Dice': 0.03, 'false_pos_vol': 2.0, 'false_neg_vol': 2.0}

# which datasets to run right now (e.g. ['DH','DeepPSMA'] if AutoPET offline)
DATASETS = ['AutoPET', 'DeepPSMA', 'DH']

# metric spec: (column, pretty, higher_is_better, diseased_only)
METRICS = [
    ('Dice',          'Dice',    True,  True),
    ('false_pos_vol', 'FP Vol',  False, False),
    ('false_neg_vol', 'FN Vol',  False, True),
]

DIAG_ORDER = ['LUNG_CANCER','LYMPHOMA','MELANOMA','NEGATIVE','PROSTATE_CANCER']
DIAG_LABEL = {'LUNG_CANCER':'Lung Cancer','LYMPHOMA':'Lymphoma','MELANOMA':'Melanoma',
              'NEGATIVE':'Negative','PROSTATE_CANCER':'Prostate Cancer'}

## 2 · Loaders (per-case metrics, per dataset/fold/SPLIT)

In [28]:
def summary_path(dataset_num, fold, split):
    return os.path.join(os.environ['nnUNet_results'], f'Dataset{dataset_num}_AutoPet',
                        TRAINER, fold, split, 'summary.json')

def convert_location_to_case_name(file_locations):
    out=[]
    for path in file_locations:
        parts=path.split('/'); out.append(f"fdg_{parts[2].replace('PETCT_','')}_{parts[3]}")
    return out

def diagnosis_lookup(metadata_csv=METADATA):
    if not os.path.exists(metadata_csv):
        print(f'[warn] {metadata_csv} not found -> FDG diagnosis NaN'); return {}
    md=pd.read_csv(metadata_csv); md['file']=convert_location_to_case_name(md['File Location'].values)
    md=md[['file','diagnosis']].drop_duplicates('file'); return dict(zip(md['file'],md['diagnosis']))

def tracer(name):
    n=name.lower(); return 'FDG' if n.startswith('fdg') else 'PSMA' if n.startswith('psma') else None

def load_arm(dataset_num, folds, split, dataset_tag, metadata_csv=METADATA):
    """One row per (fold, case): Dice, false_pos_vol, false_neg_vol + metadata."""
    dx=diagnosis_lookup(metadata_csv); rows=[]
    for fold in folds:
        sp=summary_path(dataset_num, fold, split)
        if not os.path.exists(sp): print(f'[skip] missing {sp}'); continue
        summary=json.load(open(sp))
        lpath=sp.replace('summary.json','lesion_metrics.csv'); lez=None
        if os.path.exists(lpath):
            lez=pd.read_csv(lpath); lez['filename']=lez['filename'].astype(str).str.replace('.nii.gz','',regex=False)
            lez=lez.set_index('filename')
        for case in summary['metric_per_case']:
            m=case['metrics']['1']; name=os.path.basename(case['prediction_file']).replace('.nii.gz','')
            diseased=(m['TP']+m['FN'])>0
            diag='PROSTATE_CANCER' if name.lower().startswith('psma') else dx.get(name)
            fp=fn=np.nan
            if lez is not None and name in lez.index:
                fp=lez.loc[name].get('false_pos_vol',np.nan); fn=lez.loc[name].get('false_neg_vol',np.nan)
            rows.append(dict(dataset_tag=dataset_tag, fold=fold, case_name=name, tracer=tracer(name),
                diagnosis=diag, diseased=bool(diseased), Dice=m.get('Dice',np.nan),
                false_pos_vol=fp, false_neg_vol=(np.nan if not diseased else fn)))
    df=pd.DataFrame(rows)
    if not df.empty:
        neg=df['diagnosis'].eq('NEGATIVE'); df.loc[neg,'diseased']=False; df.loc[neg,'false_neg_vol']=np.nan
    return df

## 3 · Pool the 5 random iterations (per-case mean & median)

For each case we collapse the 5 random models to one value that pairs 1:1 with FEEDS. We keep the per-case **mean** (default arm), the per-case **median** (robustness), and the per-case **SD** (for the `Random mean ± sd` display). We never average predictions or pick a single median *model*.

In [29]:
def pool_random_by_case(random_df):
    metric_cols=['Dice','false_pos_vol','false_neg_vol']
    meta=[c for c in ['dataset_tag','tracer','diagnosis','diseased'] if c in random_df.columns]
    g=random_df.groupby('case_name', dropna=False)
    out=g.agg(**{c:(c,'first') for c in meta}).reset_index()
    for m in metric_cols:
        out[m]=g[m].mean().values
        out[m+'__median']=g[m].median().values
        out[m+'__sd']=g[m].std(ddof=1).values
    out['fold']='random_summary'
    return out

def load_plan_arm(tag, arm, split):
    spec=ARMS[arm]
    df=load_arm(spec['dataset'], spec['folds'], split, dataset_tag=tag)
    return pool_random_by_case(df) if spec['agg']=='iterations' else df

## 4 · Paired tests (superiority & non-inferiority)

In [30]:
def colname(col, summary): return col if summary=='mean' else col+'__median'

def pair(f_df, c_df, f_col, c_col, diseased_only):
    fcols=['case_name',f_col]+(['diseased'] if 'diseased' in f_df else [])
    ccols=['case_name',c_col]+(['diseased'] if 'diseased' in c_df else [])
    fa=f_df[fcols].rename(columns={f_col:'_f'}); cb=c_df[ccols].rename(columns={c_col:'_c'})
    m=fa.merge(cb, on='case_name', suffixes=('_ff','_cc'))
    if diseased_only:
        for dc in ['diseased_ff','diseased_cc','diseased']:
            if dc in m: m=m[m[dc]]
    m=m.dropna(subset=['_f','_c'])
    return m['_f'].to_numpy(float), m['_c'].to_numpy(float)

def paired_superiority(feeds, comp, col, hib, dis_only, summary=SUMMARY):
    c_col=colname(col,summary); c_col=c_col if c_col in comp.columns else col
    a,b=pair(feeds,comp,col,c_col,dis_only); n=len(a)
    if n<2: return None
    diff=a-b; t,p_t=stats.ttest_rel(a,b)
    try: _,p_w=stats.wilcoxon(a,b)
    except ValueError: p_w=np.nan
    sd_col=col+'__sd'; comp_sd=comp[sd_col].mean() if sd_col in comp.columns else np.nan
    return dict(n=n, feeds=a.mean(), comp=b.mean(), comp_sd=comp_sd, delta=diff.mean(),
                t=t, p_ttest=p_t, p_wilcoxon=p_w,
                favored=('FEEDS' if ((diff.mean()<0)!=hib) else 'comparator'))

def paired_noninferiority(feeds, ref, col, hib, dis_only, margin, summary=SUMMARY, alpha=ALPHA):
    r_col=colname(col,summary); r_col=r_col if r_col in ref.columns else col
    a,b=pair(feeds,ref,col,r_col,dis_only); n=len(a)
    if n<2: return None
    diff=a-b; md=diff.mean(); se=diff.std(ddof=1)/np.sqrt(n); dfree=n-1
    if se==0:
        t_stat=np.inf; p_ni=0.0 if ((hib and md>-margin) or (not hib and md<margin)) else 1.0
    elif hib:
        t_stat=(md+margin)/se; p_ni=stats.t.sf(t_stat,dfree)
    else:
        t_stat=(md-margin)/se; p_ni=stats.t.cdf(t_stat,dfree)
    return dict(n=n, delta=md, margin=margin, t_ni=t_stat, p_noninf=p_ni, noninferior=bool(p_ni<alpha))

## 5 · Multiplicity correction — `statsmodels.stats.multitest.multipletests`

Package used: **statsmodels** (`multipletests`). `method='fdr_bh'` = Benjamini–Hochberg FDR; `method='holm'` = Holm. Both require the **entire family** of p-values, so we pass all rows of a metric at once. NaNs (e.g. Negative for Dice/FN Vol) are held out and not counted in the family size.

In [31]:
def correct_family(pvals, method=CORRECTION, alpha=ALPHA):
    p=np.asarray(pvals,float); ok=~np.isnan(p)
    corr=np.full_like(p,np.nan); rej=np.zeros(len(p),bool)
    if ok.sum()==0: return corr,rej
    reject,p_adj,_,_=multipletests(p[ok], alpha=alpha, method=method)
    corr[ok]=p_adj; rej[ok]=reject
    return corr,rej

## 6 · Build the table

Superiority + NI per (dataset, metric, group). Then correct the **superiority** Wilcoxon p-values **per metric, pooled across all datasets**. NI is left uncorrected.

In [32]:
def build_table(datasets=DATASETS, summary=SUMMARY, correction=CORRECTION, alpha=ALPHA):
    rows=[]
    for tag in datasets:
        cfg=CONFIG[tag]; sp=cfg['split']
        feeds=load_plan_arm(tag,'feeds',sp); rand=load_plan_arm(tag,'random',sp); full=load_plan_arm(tag,'full',sp)
        if cfg['layout']=='diagnosis':
            groups=[g for g in DIAG_ORDER if g in set(feeds['diagnosis'].dropna())]
        else:
            groups=[None]
        for col,pretty,hib,dis_only in METRICS:
            for grp in groups:
                f=feeds if grp is None else feeds[feeds['diagnosis']==grp]
                r=rand  if grp is None else rand [rand ['diagnosis']==grp]
                u=full  if grp is None else full [full ['diagnosis']==grp]
                sup=paired_superiority(f,r,col,hib,dis_only,summary=summary)
                if sup is None: continue      # n<2 (e.g. Negative for Dice/FN Vol)
                ni =paired_noninferiority(f,u,col,hib,dis_only,MARGINS[col],summary=summary,alpha=alpha)
                rows.append(dict(dataset=tag, metric=pretty, metric_col=col,
                    diagnosis=(DIAG_LABEL.get(grp,grp) if grp else 'Pooled'),
                    n=sup['n'], feeds=sup['feeds'], random_mean=sup['comp'], random_sd=sup['comp_sd'],
                    delta=sup['delta'], p_wilcoxon=sup['p_wilcoxon'], p_ttest=sup['p_ttest'],
                    favored=sup['favored'], ni_p=(ni['p_noninf'] if ni else np.nan),
                    ni_pass=(ni['noninferior'] if ni else np.nan)))
    tab=pd.DataFrame(rows)
    if tab.empty: return tab
    corr_col=f'p_wilcoxon_{correction}'; tab[corr_col]=np.nan; tab['sig_'+correction]=False
    print(f'SUPERIORITY family sizes (correction denominator, method={correction}):')
    for col in tab['metric_col'].unique():
        mask=tab['metric_col']==col; pretty=tab.loc[mask,'metric'].iloc[0]
        print(f'   {pretty}: {int(mask.sum())} tests')
        cp,rej=correct_family(tab.loc[mask,'p_wilcoxon'].values, method=correction, alpha=alpha)
        tab.loc[mask,corr_col]=cp; tab.loc[mask,'sig_'+correction]=rej
    print(f'   SUPERIORITY total: {len(tab)} tests (NI excluded from correction)')
    return tab

tab = build_table()
tab

SUPERIORITY family sizes (correction denominator, method=fdr_bh):
   Dice: 6 tests
   FP Vol: 7 tests
   FN Vol: 6 tests
   SUPERIORITY total: 19 tests (NI excluded from correction)


,dataset,metric,metric_col,diagnosis,n,feeds,random_mean,random_sd,delta,p_wilcoxon,p_ttest,favored,ni_p,ni_pass,p_wilcoxon_fdr_bh,sig_fdr_bh
0,AutoPET,Dice,Dice,Lung Cancer,35,0.725535,0.735002,0.035343,-0.009466,3.340676e-01,0.305175,comparator,3.316095e-01,False,4.469073e-01,False
1,AutoPET,Dice,Dice,Lymphoma,30,0.685861,0.655709,0.081239,0.030152,4.320019e-02,0.113643,FEEDS,6.986276e-01,False,1.296006e-01,False
2,AutoPET,Dice,Dice,Melanoma,32,0.656070,0.652815,0.074587,0.003256,5.999359e-01,0.894024,FEEDS,1.300039e-01,False,5.999359e-01,False
3,AutoPET,Dice,Dice,Prostate Cancer,110,0.544190,0.551607,0.057652,-0.007417,3.350729e-01,0.408147,comparator,6.990710e-01,False,4.469073e-01,False
4,AutoPET,FP Vol,false_pos_vol,Lung Cancer,35,1.962482,3.298719,3.589084,-1.336237,2.851193e-03,0.120527,FEEDS,6.429950e-04,True,3.991670e-03,True
5,AutoPET,FP Vol,false_pos_vol,Lymphoma,30,2.152711,4.334369,2.178592,-2.181658,3.800932e-04,0.039026,FEEDS,1.907715e-07,True,6.651632e-04,True
6,AutoPET,FP Vol,false_pos_vol,Melanoma,32,4.070547,5.093276,2.869127,-1.022730,6.478708e-03,0.120337,FEEDS,1.147231e-03,True,7.558492e-03,True
7,AutoPET,FP Vol,false_pos_vol,Negative,102,5.604558,8.109204,4.915314,-2.504646,6.357151e-15,0.001108,FEEDS,1.079500e-08,True,2.225003e-14,True
8,AutoPET,FP Vol,false_pos_vol,Prostate Cancer,122,10.496115,10.173164,2.952887,0.322952,2.309669e-07,0.721511,comparator,8.617966e-03,True,5.389227e-07,True
9,AutoPET,FN Vol,false_neg_vol,Lung Cancer,35,14.135844,15.014464,3.643070,-0.878620,1.246850e-02,0.778083,FEEDS,5.527387e-01,False,7.481100e-02,False


## 7 · Formatted table (matches the manuscript figure)

In [33]:
def format_table(tab, correction=CORRECTION):
    corr_col=f'p_wilcoxon_{correction}'
    def fmt_p(x): return '--' if pd.isna(x) else (f'{x:.2e}' if x<1e-3 else f'{x:.3f}')
    def fmt_v(x,metric): return '--' if pd.isna(x) else (f'{x:.3f}' if metric=='Dice' else f'{x:.2f}')
    out=[]
    for _,r in tab.iterrows():
        out.append({'Dataset':r['dataset'],'Metric':r['metric'],'Diagnosis':r['diagnosis'],
            'FEEDS':fmt_v(r['feeds'],r['metric']),
            'Random':f"{fmt_v(r['random_mean'],r['metric'])} ± {fmt_v(r['random_sd'],r['metric'])}",
            'Δ':('+' if r['delta']>=0 else '')+fmt_v(r['delta'],r['metric']),
            'Wilc p (raw)':fmt_p(r['p_wilcoxon']),
            f'Wilc p ({("FDR" if correction=="fdr_bh" else correction.upper())})':fmt_p(r[corr_col]),
            'NI vs 100%':('✓' if r['ni_pass'] in (True,1) else ('✗' if r['ni_pass'] in (False,0) else '--'))})
    return pd.DataFrame(out)

disp = format_table(tab)
disp

,Dataset,Metric,Diagnosis,FEEDS,Random,Δ,Wilc p (raw),Wilc p (FDR),NI vs 100%
0,AutoPET,Dice,Lung Cancer,0.726,0.735 ± 0.035,-0.009,0.334,0.447,✗
1,AutoPET,Dice,Lymphoma,0.686,0.656 ± 0.081,+0.030,0.043,0.130,✗
2,AutoPET,Dice,Melanoma,0.656,0.653 ± 0.075,+0.003,0.600,0.600,✗
3,AutoPET,Dice,Prostate Cancer,0.544,0.552 ± 0.058,-0.007,0.335,0.447,✗
4,AutoPET,FP Vol,Lung Cancer,1.96,3.30 ± 3.59,-1.34,0.003,0.004,✓
5,AutoPET,FP Vol,Lymphoma,2.15,4.33 ± 2.18,-2.18,3.80e-04,6.65e-04,✓
6,AutoPET,FP Vol,Melanoma,4.07,5.09 ± 2.87,-1.02,0.006,0.008,✓
7,AutoPET,FP Vol,Negative,5.60,8.11 ± 4.92,-2.50,6.36e-15,2.23e-14,✓
8,AutoPET,FP Vol,Prostate Cancer,10.50,10.17 ± 2.95,+0.32,2.31e-07,5.39e-07,✓
9,AutoPET,FN Vol,Lung Cancer,14.14,15.01 ± 3.64,-0.88,0.012,0.075,✗


## 8 · Save

In [34]:
os.makedirs('outputs', exist_ok=True)
tab.to_csv(f'outputs/comparison_table_full_{SUMMARY}_{CORRECTION}.csv', index=False)
disp.to_csv(f'outputs/comparison_table_display_{SUMMARY}_{CORRECTION}.csv', index=False)
print('saved to outputs/')

saved to outputs/


## Notes

- **Change SPLIT** in the `CONFIG` cell (§1) — each dataset's `split` key is the only thing that differs between AutoPET / Deep-PSMA / DH. To skip a dataset, drop it from `DATASETS`.
- **mean vs median**: set `SUMMARY='median'` and re-run to get the robustness version of the whole table.
- **Correction family**: per metric, pooled across all datasets, on the Wilcoxon p-value. With all three datasets that's Dice 6, FP Vol 7, FN Vol 6 = 19 superiority tests; NI's 19 are a separate family and stay uncorrected.
- **Package**: correction is done by `statsmodels.stats.multitest.multipletests` (`fdr_bh` / `holm`).